# MATMED Phase 3: Vision-Agent Integration
> **New in Phase 3:** Each molecule triggers a `simulate_reaction_video()` call, producing a `(50, 16)` time-series tensor that is processed by the `VisionTemporalTransformer` and **cross-attended** with the molecular embedding inside the R-Agent to improve yield prediction.

**New metrics logged per episode:**
- `vision_variance` — how chaotic/noisy the molecule's reaction signal is
- `yield_sa_corr` — Pearson correlation between predicted yield and SA score

## 1. Install Dependencies & Clone Repo

In [ ]:
!pip install -q rdkit-pypi torch transformers numpy pandas matplotlib "kagglehub[pandas-datasets]"

import os
if not os.path.exists('matmed'):
    !git clone https://github.com/scott2srikanth/matmed.git
else:
    !cd matmed && git fetch --all

%cd matmed
!git checkout phase3
!git pull origin phase3
print('\n\u2705 Repository ready on phase3 branch!')

## 2. Test Vision Agent — Simulate Reaction Video

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from vision_agent import simulate_reaction_video, VisionTemporalTransformer

# Aspirin: SA~2.5, MW~180, logP~1.2
# Normalized: [norm_sa=0.83, norm_mw=0.7, norm_lp=0.78]
easy_mol_feats  = np.array([0.83, 0.70, 0.78], dtype=np.float32)  # Easy to synthesize
hard_mol_feats  = np.array([0.10, 0.20, 0.30], dtype=np.float32)  # Hard to synthesize

easy_video = simulate_reaction_video(easy_mol_feats)
hard_video = simulate_reaction_video(hard_mol_feats)

print(f'Easy mol video shape : {easy_video.shape}  variance={torch.var(easy_video):.4f}')
print(f'Hard mol video shape : {hard_video.shape}  variance={torch.var(hard_video):.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(easy_video.numpy(), alpha=0.6)
axes[0].set_title('Easy Molecule Reaction Video (low noise)')
axes[0].set_xlabel('Timestep'); axes[0].set_ylabel('Signal')
axes[1].plot(hard_video.numpy(), alpha=0.6)
axes[1].set_title('Hard Molecule Reaction Video (high noise/chaos)')
axes[1].set_xlabel('Timestep')
plt.tight_layout(); plt.show()

## 3. Test VisionTemporalTransformer

In [ ]:
vtt = VisionTemporalTransformer(input_dim=28, embed_dim=128, num_layers=3, num_heads=4)
print(f'VisionTemporalTransformer params: {sum(p.numel() for p in vtt.parameters()):,}')

# Forward pass with batched easy molecule video
batched = easy_video.unsqueeze(0)   # (1, 50, 16)
emb = vtt(batched)
print(f'Output embedding shape: {emb.shape}   (expected: torch.Size([1, 128]))')
assert emb.shape == (1, 128), 'Shape mismatch!'
print('\u2705 VisionTemporalTransformer OK!')

## 4. Test R-Agent with Vision Cross-Attention

In [ ]:
from reaction_agent import ReactionAgent, compute_reaction_features

agent = ReactionAgent(use_vision=True)
print(f'R-Agent (Phase 3) params: {sum(p.numel() for p in agent.parameters()):,}')

test_smiles = [
    'CC(=O)Oc1ccccc1C(=O)O',  # Aspirin
    'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',  # Caffeine
    'CC12CCC3C(C1CCC2O)CCC4=CC(=O)CCC34C',  # Testosterone
]

print('\nPredictions WITH vision cross-attention:')
for smi in test_smiles:
    feats = compute_reaction_features(smi)
    vis   = simulate_reaction_video(feats)    # (50, 16)
    score, emb = agent(smi, vision_seq=vis)
    vis_var = float(torch.var(vis).item())
    print(f'  yield={score:.4f}  vis_var={vis_var:.4f}  {smi[:40]}')

print('\nPredictions WITHOUT vision (backward compat check):')
for smi in test_smiles:
    score, emb = agent(smi)
    print(f'  yield={score:.4f}  {smi[:40]}')

## 5. Full Phase 3 Training
Run the complete MATMED RL loop with vision signals — logs `vision_variance` and `yield_sa_corr` each episode.

In [ ]:
from reward import RewardConfig
from train_matmed import MATMEDRunner

reward_cfg = RewardConfig(alpha=1.0, beta=1.0, gamma=1.0, delta=1.0)

runner = MATMEDRunner(
    reward_config=reward_cfg,
    num_pretrain_epochs=20,
    seed=42,
    use_chemberta=False,
)

runner.train(
    num_episodes=30,
    steps_per_episode=8,
    save_csv='matmed_phase3_metrics.csv',
)

## 6. Analyse Phase 3 Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('matmed_phase3_metrics.csv')
print('Columns:', list(df.columns))
print(df.tail(10).to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('MATMED Phase 3 — Vision-Agent Training Metrics', fontsize=14, fontweight='bold')

df['avg_reward'].plot(ax=axes[0,0],  title='Avg Reward',         color='steelblue')
df['best_reward'].plot(ax=axes[0,1], title='Best Reward',        color='darkorange')
df['pct_valid'].plot(ax=axes[0,2],   title='SMILES Validity %',  color='green')
df['diversity'].plot(ax=axes[1,0],   title='Chemical Diversity', color='purple')
df['vision_variance'].plot(ax=axes[1,1], title='Vision Variance (avg per episode)', color='red')
df['yield_sa_corr'].plot(ax=axes[1,2],  title='Yield ↔ SA Score Correlation', color='brown')

for ax in axes.flat:
    ax.set_xlabel('Episode'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('matmed_phase3_plot.png', dpi=150)
plt.show()
print(f'Plot saved to matmed_phase3_plot.png')

## 7. Visualise Best Molecule

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

best = runner.best_smiles or 'CC(=O)Oc1ccccc1C(=O)O'
mol  = Chem.MolFromSmiles(best)
if mol:
    display(Draw.MolToImage(mol, size=(400, 300)))
    print(f'Best SMILES : {best}')
    print(f'Best reward : {runner.best_reward:.4f}')
else:
    print('Falling back to Aspirin')
    display(Draw.MolToImage(Chem.MolFromSmiles('CC(=O)Oc1ccccc1C(=O)O'), size=(400, 300)))

## 8. Phase 3 Ablation Study
Quantify the contribution of Vision and Uncertainty by running 3 variants:
| Config | Vision | MC Dropout lambda |
|--------|--------|-------------------|
| Full MATMED | Yes | 0.1 |
| No Vision | No | 0.0 |
| No Uncertainty | Yes | 0.0 |

In [ ]:
!python phase3_ablation.py

## 9. Plot Phase 3 Ablation Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

dfs = {
    'Full MATMED':    pd.read_csv('phase3_ablation_full_matmed.csv'),
    'No Vision':      pd.read_csv('phase3_ablation_no_vision.csv'),
    'No Uncertainty': pd.read_csv('phase3_ablation_no_uncertainty.csv'),
}
colors = {'Full MATMED': 'steelblue', 'No Vision': 'tomato', 'No Uncertainty': 'darkorange'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Phase 3 Ablation: Vision vs No-Vision vs No-Uncertainty',
             fontsize=14, fontweight='bold')

panels = [
    ('avg_reward',      'Avg Reward',           axes[0,0]),
    ('best_reward',     'Best Reward',          axes[0,1]),
    ('pct_valid',       'SMILES Validity %',    axes[0,2]),
    ('diversity',       'Chemical Diversity',   axes[1,0]),
    ('vision_variance', 'Vision Variance',      axes[1,1]),
    ('yield_sa_corr',   'Yield <-> SA Corr',    axes[1,2]),
]

for col, title, ax in panels:
    for name, df in dfs.items():
        vals = df[col] if col in df.columns else [0]*len(df)
        ax.plot(df['episode'], vals, label=name,
                color=colors[name], linewidth=2)
    ax.set_title(title); ax.set_xlabel('Episode'); ax.grid(alpha=0.3)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('phase3_ablation_plot.png', dpi=150)
plt.show()

# Summary table (avg of last 5 episodes)
print('\n--- Last-5-Episode Summary ---')
rows = []
for name, df in dfs.items():
    tail = df.tail(5)
    rows.append({
        'Config':         name,
        'avg_reward':     round(tail['avg_reward'].mean(),  4),
        'best_reward':    round(tail['best_reward'].max(),  4),
        'pct_valid':      round(tail['pct_valid'].mean(),   2),
        'yield_sa_corr':  round(tail['yield_sa_corr'].mean(), 4)
            if 'yield_sa_corr' in tail.columns else 'N/A',
    })
print(pd.DataFrame(rows).set_index('Config').to_string())